# 🚀 Kronos-base Cloud GPU Inference Server
Run Kronos foundation model inference on Google Colab GPU (T4 / A100) and connect directly to your Laravel application.

### Instructions:
1. **Runtime Type**: Go to `Runtime` > `Change runtime type` > select **T4 GPU** (free).
2. Run **Step 1** to confirm GPU availability.
3. Run **Step 2** to clone official Kronos architecture and your custom GitHub repository.
4. Run **Step 3** to install requirements.
5. Run **Step 4** to start the FastAPI server with an **ngrok** tunnel and get your `KRONOS_API_URL`.

In [ ]:
# Step 1: Verify Colab GPU acceleration
!nvidia-smi

In [ ]:
# Step 2: Clone Kronos architecture and your custom microservice repository
# 2a. Official Kronos model code
!rm -rf /content/Kronos
!git clone https://github.com/shiyu-coder/Kronos.git /content/Kronos

# 2b. Clone your standalone Kronos microservice repository
# REPLACE with your new GitHub repo URL if you created a dedicated one:
YOUR_GITHUB_REPO = "https://github.com/janakkafle/kronos-service.git"

!rm -rf /content/kronos_service
!git clone $YOUR_GITHUB_REPO /content/kronos_service

# If you haven't pushed to a dedicated repo yet, you can also upload files directly or clone the subfolder.

In [ ]:
# Step 3: Install all required Python packages
!pip install -q fastapi uvicorn pydantic pyngrok transformers accelerate safetensors pandas numpy requests

In [ ]:
# Step 4: Configure Ngrok with static domain and launch FastAPI server
import sys
import os

# Add Kronos official repository & microservice to Python path
sys.path.append('/content/Kronos')
sys.path.append('/content/kronos_service')

# Dedicated static domain credentials
NGROK_AUTH_TOKEN = "3M59t38Z8aG23qQfLsCx5hm0PKK_2UZ26SHAJhwVyniPYpV6u"
NGROK_DOMAIN = "moisten-vacation-wizard.ngrok-free.dev"

from pyngrok import ngrok
ngrok.kill()
if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Connect static tunnel to FastAPI on port 8000
try:
    tunnel = ngrok.connect(8000, domain=NGROK_DOMAIN)
except Exception as exc:
    print(f"Domain connect notice ({exc}), trying with url param...")
    try:
        tunnel = ngrok.connect(8000, url=f"https://{NGROK_DOMAIN}")
    except Exception:
        tunnel = ngrok.connect(8000)

public_url = tunnel.public_url

print("\n" + "="*65)
print("⚡ KRONOS GPU INFERENCE SERVER IS ONLINE (STATIC DOMAIN)!")
print(f"Public Static URL : {public_url}")
print(f"Keep-Alive Poke URL: {public_url}/poke")
print("="*65)
print("Your Laravel .env is pre-configured with:")
print(f"KRONOS_API_URL={public_url}")
print("="*65 + "\n")

# Launch FastAPI server
%cd /content/kronos_service
!uvicorn app:app --host 0.0.0.0 --port 8000


### 🛡️ Colab Keep-Alive Tips (Prevent Disconnection):
1. **Colab Browser Extension**: Run your browser extension to keep the tab awake.
2. **Zero-Load Heartbeat Poke**: An external cron or script pings `https://moisten-vacation-wizard.ngrok-free.dev/poke` every 10 seconds. This generates continuous safe HTTP traffic with 0% GPU load!
3. **Browser Console Snippet**: If needed, press `F12` on the Colab tab, go to **Console**, paste this, and press Enter:
```javascript
setInterval(() => {
  console.log('Keep-alive ping');
  document.querySelector('colab-connect-button')?.click();
}, 60000);
```